# Mixture of Experts (MoE) - 실습 코드 1: MoE Layer 구현 (PyTorch)

- Tutorial ID: `expand-moe`
- Tutorial: Mixture of Experts (MoE)
- Section ID: `expand-moe-code-1`
- Section: 실습 코드 1: MoE Layer 구현 (PyTorch)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: MoE Layer 구현 (PyTorch)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoELayer(nn.Module):
        def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.top_k = top_k
        self.num_experts = num_experts
        
        # Router
        self.router = nn.Linear(d_model, num_experts)
        
        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.SiLU(),
                nn.Linear(d_ff, d_model),
            ) for _ in range(num_experts)
        ])
        
        # Shared Expert (DeepSeek-style)
        self.shared_expert = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.SiLU(),
            nn.Linear(d_ff, d_model),
        )
        
        def forward(self, x):
        batch, seq, d = x.shape
        x_flat = x.view(-1, d)
        
        # Router scores
                logits = self.router(x_flat)
        top_k_vals, top_k_idx = torch.topk(logits, self.top_k, dim=-1)
                top_k_weights = F.softmax(top_k_vals, dim=-1)
        
        # Expert outputs
                output = torch.zeros_like(x_flat)
                for i in range(self.num_experts):
                        mask = (top_k_idx == i).any(dim=-1)
            if mask.any():
                                expert_out = self.experts[i](x_flat[mask])
                                for k in range(self.top_k):
                                        expert_mask = (top_k_idx[mask, k] == i)
                    if expert_mask.any():
                        w = top_k_weights[mask, k][expert_mask].unsqueeze(-1)
                        indices = mask.nonzero()[expert_mask].squeeze()
                        output[indices] += w * expert_out[expert_mask]
        
        # Add shared expert
                output = output + self.shared_expert(x_flat)
        return output.view(batch, seq, d)

# 테스트
moe = MoELayer(d_model=512, d_ff=2048, num_experts=8, top_k=2)
x = torch.randn(2, 10, 512)
out = moe(x)
print(f"MoE output: {out.shape}")